# Deubner (1975): k–ω Diagram Construction / k–ω 다이어그램 구성

**Paper**: Deubner, F.-L. 1975, A&A, 44, 371–375
"Observations of Low Wavenumber Nonradial Eigenmodes of the Sun"

## Objectives / 목표

1. **Simulate solar oscillation data** $v(x, t)$ with multiple trapped p-mode ridges
   태양 진동 데이터 $v(x, t)$를 다중 갇힌 p-mode 능선으로 시뮬레이션

2. **Construct the k–ω diagnostic diagram** via 2D Fourier transform
   2D Fourier 변환으로 k–ω 진단 다이어그램 구성

3. **Compare with Ulrich's (1970) theoretical dispersion relation**
   Ulrich (1970)의 이론적 분산 관계와 비교

4. **Reproduce the key observational result**: discrete ridges in k–ω space
   핵심 관측 결과 재현: k–ω 공간에서의 이산 능선

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

plt.rcParams.update({
    "figure.figsize": (10, 8),
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
})

## 1. Solar Model: Sound Speed Profile / 태양 모델: 음속 프로파일

Ulrich's theory requires knowledge of how the sound speed $c(r)$ varies with depth.
We use a simplified polytropic model to compute the sound speed profile:

Ulrich의 이론은 음속 $c(r)$이 깊이에 따라 어떻게 변하는지 알아야 한다.
단순화된 polytrope 모델을 사용하여 음속 프로파일을 계산한다:

$$c^2(r) = \frac{\gamma k_B T(r)}{\mu m_H}$$

For a polytrope of index $n_p$, the temperature increases linearly with depth below the photosphere:

Polytrope 지수 $n_p$인 모델에서, 온도는 광구 아래 깊이에 따라 선형적으로 증가한다:

$$T(r) = T_{\text{phot}} \left[1 + \frac{R_\odot - r}{d_{\text{cz}}}\left(\frac{T_{\text{base}}}{T_{\text{phot}}} - 1\right)\right]$$

In [ ]:
# Solar constants
R_SUN = 696.0  # Mm (solar radius)
T_PHOT = 5778.0  # K (photosphere temperature)
T_BASE = 2.3e6  # K (base of convection zone)
D_CZ = 0.287 * R_SUN  # Mm (convection zone depth ~ 200 Mm)
GAMMA = 5.0 / 3.0  # adiabatic index
MU = 0.6  # mean molecular weight (fully ionized H/He mix)
K_B = 1.381e-23  # J/K
M_H = 1.673e-27  # kg


def sound_speed(r):
    """Compute sound speed at radius r (Mm) using a polytropic model.

    Args:
        r: Radial distance from center in Mm.

    Returns:
        Sound speed in Mm/s.
    """
    depth = R_SUN - r
    depth = np.clip(depth, 0, D_CZ)
    T = T_PHOT * (1.0 + (depth / D_CZ) * (T_BASE / T_PHOT - 1.0))
    c = np.sqrt(GAMMA * K_B * T / (MU * M_H))  # m/s
    return c * 1e-6  # convert to Mm/s


# Plot sound speed profile
r_arr = np.linspace(R_SUN - D_CZ, R_SUN, 500)
c_arr = sound_speed(r_arr)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot((R_SUN - r_arr), c_arr * 1e3, "b-", linewidth=2)
ax.set_xlabel("Depth below photosphere / 광구 아래 깊이 (Mm)")
ax.set_ylabel("Sound speed / 음속 (km/s)")
ax.set_title("Sound Speed Profile in the Convection Zone\n대류층 음속 프로파일")
ax.grid(True, alpha=0.3)
ax.invert_xaxis()
plt.tight_layout()
plt.show()
print(f"Photosphere c = {sound_speed(R_SUN)*1e3:.1f} km/s")
print(f"CZ base c = {sound_speed(R_SUN - D_CZ)*1e3:.1f} km/s")

## 2. Theoretical p-mode Dispersion Relation / 이론적 p-mode 분산 관계

For a given horizontal wavenumber $k_h$, the lower turning point $r_t$ is where refraction turns the wave back toward the surface. This occurs where:

수평 파수 $k_h$가 주어지면, 하부 반전점 $r_t$는 굴절에 의해 파동이 표면으로 되돌아가는 깊이다. 이는 다음 조건에서 발생한다:

$$\frac{\omega}{k_h} = c(r_t)$$

The resonance condition requires an integer number of half-wavelengths to fit vertically between the upper reflecting surface ($r \approx R_\odot$) and the lower turning point:

공진 조건은 상부 반사면($r \approx R_\odot$)과 하부 반전점 사이에 정수 반파장이 맞춰질 것을 요구한다:

$$\int_{r_t}^{R_\odot} k_r(r) \, dr = \left(n + \frac{1}{2}\right)\pi, \quad n = 0, 1, 2, \ldots$$

where / 여기서

$$k_r(r) = \sqrt{\frac{\omega^2}{c(r)^2} - k_h^2\left(\frac{1}{r^2}\right)\cdot r^2} \approx \sqrt{\frac{\omega^2}{c(r)^2} - \frac{\ell(\ell+1)}{r^2}}$$

We solve this numerically for each radial order $n$ to obtain the dispersion curves $\omega(\ell, n)$.

각 radial order $n$에 대해 이를 수치적으로 풀어 분산 곡선 $\omega(\ell, n)$을 얻는다.

In [ ]:
from scipy.optimize import brentq
from scipy.integrate import quad


def turning_point_radius(omega, ell):
    """Find the lower turning point radius for given omega and ell.

    The turning point is where omega/k_h = c(r), i.e., where
    omega * r / sqrt(ell*(ell+1)) = c(r).

    Args:
        omega: Angular frequency in rad/s.
        ell: Spherical harmonic degree.

    Returns:
        Turning point radius in Mm, or None if no turning point exists.
    """
    if ell == 0:
        return 0.0

    L = np.sqrt(ell * (ell + 1))

    def func(r):
        return omega * r / L - sound_speed(r)

    r_min = R_SUN - D_CZ + 0.1
    r_max = R_SUN - 0.1

    f_min = func(r_min)
    f_max = func(r_max)

    if f_min * f_max > 0:
        return None

    return brentq(func, r_min, r_max)


def phase_integral(omega, ell, r_t):
    """Compute the radial phase integral from turning point to surface.

    Args:
        omega: Angular frequency in rad/s.
        ell: Spherical harmonic degree.
        r_t: Lower turning point radius in Mm.

    Returns:
        Phase integral value.
    """
    L2 = ell * (ell + 1)

    def integrand(r):
        c = sound_speed(r)
        kr2 = (omega / c) ** 2 - L2 / r**2
        if kr2 <= 0:
            return 0.0
        return np.sqrt(kr2)

    result, _ = quad(integrand, r_t + 0.01, R_SUN - 0.01, limit=200)
    return result


def find_eigenfrequency(ell, n, omega_range=(1e-3, 6e-2)):
    """Find the eigenfrequency for given (ell, n) by solving the resonance condition.

    Args:
        ell: Spherical harmonic degree.
        n: Radial order.
        omega_range: Search range for omega in rad/s.

    Returns:
        Eigenfrequency in rad/s, or None if not found.
    """
    target_phase = (n + 0.5) * np.pi

    def residual(omega):
        r_t = turning_point_radius(omega, ell)
        if r_t is None:
            return -target_phase
        phi = phase_integral(omega, ell, r_t)
        return phi - target_phase

    omega_lo, omega_hi = omega_range
    try:
        # Search for sign change
        n_search = 200
        omegas = np.linspace(omega_lo, omega_hi, n_search)
        residuals = np.array([residual(o) for o in omegas])

        for i in range(len(residuals) - 1):
            if residuals[i] * residuals[i + 1] < 0:
                return brentq(residual, omegas[i], omegas[i + 1], xtol=1e-7)
    except (ValueError, RuntimeError):
        pass
    return None


print("Testing eigenfrequency calculation...")
test_omega = find_eigenfrequency(ell=100, n=1)
if test_omega is not None:
    nu = test_omega / (2 * np.pi)  # Hz
    period = 1.0 / nu  # seconds
    print(f"  ell=100, n=1: ω = {test_omega:.4e} rad/s, "
          f"ν = {nu*1e3:.2f} mHz, P = {period:.0f} s")
else:
    print("  No solution found for test case.")

## 3. Compute Theoretical Dispersion Curves / 이론적 분산 곡선 계산

Compute $\omega(\ell, n)$ for several radial orders $n = 0, 1, 2, 3, 4$ across a range of $\ell$ values,
matching the range Deubner could observe ($k_h \lesssim 1.0$ Mm$^{-1}$, i.e., $\ell \lesssim 700$).

여러 radial order $n = 0, 1, 2, 3, 4$에 대해 $\ell$ 범위에 걸쳐 $\omega(\ell, n)$을 계산한다.
Deubner가 관측할 수 있었던 범위($k_h \lesssim 1.0$ Mm$^{-1}$, 즉 $\ell \lesssim 700$)에 맞춘다.

In [ ]:
# Compute dispersion curves for n = 0..4
ell_values = np.arange(20, 700, 10)
n_orders = [0, 1, 2, 3, 4]
dispersion_curves = {}

for n in n_orders:
    kh_list = []
    omega_list = []
    for ell in ell_values:
        omega = find_eigenfrequency(ell, n)
        if omega is not None:
            kh = np.sqrt(ell * (ell + 1)) / R_SUN  # Mm^-1
            nu = omega / (2 * np.pi) * 1e3  # mHz
            if 0.5 < nu < 10.0:  # physically reasonable range
                kh_list.append(kh)
                omega_list.append(nu)
    dispersion_curves[n] = (np.array(kh_list), np.array(omega_list))
    print(f"n={n}: {len(kh_list)} points computed")

print("\nDispersion curves computed successfully.")

## 4. Simulate Observed Velocity Field v(x, t) / 관측 속도장 시뮬레이션

We simulate what Deubner would have observed: a velocity field $v(x, t)$ along a 300 arcsec slit,
sampled every 1.0 arcsec in space and every 110 s in time. The signal is a superposition of
p-mode oscillations riding on the theoretical dispersion ridges, plus noise.

Deubner가 관측했을 것을 시뮬레이션한다: 300 arcsec 슬릿을 따른 속도장 $v(x, t)$를
공간 1.0 arcsec, 시간 110초 간격으로 샘플링한다. 신호는 이론적 분산 능선 위의
p-mode 진동들의 중첩이며, 잡음이 추가된다.

$$v(x, t) = \sum_{n, k_h} A_n \cos(k_h x - \omega_{n}(k_h) t + \phi_{n,k_h}) + \text{noise}$$

In [ ]:
# Observation parameters matching Deubner's setup
ARCSEC_TO_MM = 0.725e-3  # 1 arcsec ≈ 0.725 Mm on the Sun
SLIT_LENGTH_ARCSEC = 300  # arcsec
SCAN_STEP = 1.0  # arcsec
SCAN_PERIOD = 110.0  # seconds
OBS_DURATION = 3 * 3600 + 55 * 60  # 3h55m in seconds (longer set)

# Spatial and temporal grids
nx = int(SLIT_LENGTH_ARCSEC / SCAN_STEP)  # 300 spatial points
nt = int(OBS_DURATION / SCAN_PERIOD)  # ~128 time steps

x_arcsec = np.arange(nx) * SCAN_STEP  # arcsec
x_mm = x_arcsec * ARCSEC_TO_MM  # Mm
t_sec = np.arange(nt) * SCAN_PERIOD  # seconds

print(f"Spatial grid: {nx} points, dx = {SCAN_STEP} arcsec = {SCAN_STEP * ARCSEC_TO_MM * 1e3:.2f} km")
print(f"Temporal grid: {nt} points, dt = {SCAN_PERIOD} s")
print(f"Slit length: {SLIT_LENGTH_ARCSEC} arcsec = {SLIT_LENGTH_ARCSEC * ARCSEC_TO_MM:.1f} Mm")
print(f"Observation duration: {OBS_DURATION/3600:.2f} hours")

In [ ]:
rng = np.random.default_rng(42)

# Build v(x, t) by superposing modes on each dispersion ridge
X, T = np.meshgrid(x_mm, t_sec, indexing="ij")  # shape (nx, nt)
v = np.zeros_like(X)

# Amplitude decreases with radial order (higher n = weaker)
amp_scale = {0: 1.0, 1: 0.8, 2: 0.6, 3: 0.4, 4: 0.25}

for n, (kh_arr, nu_arr) in dispersion_curves.items():
    if len(kh_arr) == 0:
        continue
    # Sample modes along the ridge
    n_modes = min(40, len(kh_arr))
    indices = np.linspace(0, len(kh_arr) - 1, n_modes, dtype=int)
    for idx in indices:
        kh = kh_arr[idx]  # Mm^-1
        omega = nu_arr[idx] * 2 * np.pi * 1e-3  # convert mHz -> rad/s
        phase = rng.uniform(0, 2 * np.pi)
        amp = amp_scale.get(n, 0.2) * rng.uniform(0.5, 1.5)
        # Add finite lifetime via exponential damping envelope
        t_center = rng.uniform(0.2, 0.8) * OBS_DURATION
        lifetime = rng.uniform(1500, 4000)  # seconds
        envelope = np.exp(-((T - t_center) ** 2) / (2 * lifetime**2))
        v += amp * envelope * np.cos(kh * X - omega * T + phase)

# Add noise (SNR ~ 3)
signal_rms = np.std(v)
noise_level = signal_rms / 3.0
v += rng.normal(0, noise_level, v.shape)

print(f"Signal RMS: {signal_rms:.3f}")
print(f"Noise level: {noise_level:.3f}")
print(f"v(x,t) shape: {v.shape} = (nx, nt)")

# Visualize v(x, t)
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.pcolormesh(
    t_sec / 60, x_arcsec, v,
    cmap="seismic", shading="auto",
    vmin=-np.percentile(np.abs(v), 99),
    vmax=np.percentile(np.abs(v), 99),
)
ax.set_xlabel("Time / 시간 (min)")
ax.set_ylabel("Position along slit / 슬릿 위치 (arcsec)")
ax.set_title("Simulated velocity field v(x, t)\n시뮬레이션된 속도장 v(x, t)")
plt.colorbar(im, ax=ax, label="Velocity (arb. units)")
plt.tight_layout()
plt.show()

## 5. Construct the k–ω Diagram via 2D FFT / 2D FFT로 k–ω 다이어그램 구성

This is the core of Deubner's analysis. We apply a 2D Fourier transform to $v(x, t)$
and compute the power spectrum $P(k_h, \omega)$.

이것이 Deubner 분석의 핵심이다. $v(x, t)$에 2D Fourier 변환을 적용하여
파워 스펙트럼 $P(k_h, \omega)$를 계산한다.

$$P(k_h, \omega) = \left|\hat{V}(k_h, \omega)\right|^2 = \left|\iint v(x,t)\,e^{-i(k_h x - \omega t)}\,dx\,dt\right|^2$$

We apply a Hamming window before FFT to reduce spectral leakage, following Deubner's procedure.

Deubner의 절차를 따라 FFT 전에 Hamming window를 적용하여 스펙트럼 누설을 줄인다.

In [ ]:
# Apply 2D Hamming window
window_x = np.hamming(nx)
window_t = np.hamming(nt)
window_2d = np.outer(window_x, window_t)
v_windowed = v * window_2d

# 2D FFT
V_fft = np.fft.fft2(v_windowed)
P_kw = np.abs(V_fft) ** 2

# Frequency and wavenumber axes
# Spatial: x in Mm -> k in Mm^-1
dx_mm = SCAN_STEP * ARCSEC_TO_MM  # Mm
k_arr = np.fft.fftfreq(nx, d=dx_mm) * 2 * np.pi  # rad/Mm -> use cycles: fftfreq gives cycles/Mm
# Actually use fftfreq which gives cycles per unit, multiply by 2*pi for angular wavenumber
# But Deubner uses k in Mm^-1 (not angular), so keep as cycles/Mm
k_arr = np.fft.fftfreq(nx, d=dx_mm)  # cycles per Mm

# Temporal: t in seconds -> frequency in mHz
dt_s = SCAN_PERIOD
freq_arr = np.fft.fftfreq(nt, d=dt_s) * 1e3  # mHz

# Take only positive frequencies and wavenumbers
k_pos = k_arr[:nx // 2]
freq_pos = freq_arr[:nt // 2]
P_pos = P_kw[:nx // 2, :nt // 2]

# Smooth with a small Gaussian kernel for better visualization
from scipy.ndimage import gaussian_filter
P_smooth = gaussian_filter(P_pos, sigma=[1.5, 1.5])

print(f"k range: {k_pos[1]:.4f} to {k_pos[-1]:.4f} Mm^-1")
print(f"freq range: {freq_pos[1]:.4f} to {freq_pos[-1]:.4f} mHz")
print(f"Δk = {k_pos[1]:.4f} Mm^-1")
print(f"Δν = {freq_pos[1]:.4f} mHz")

## 6. Plot the k–ω Diagram / k–ω 다이어그램 출력

The main result: the diagnostic $k$–$\omega$ diagram showing discrete ridges,
reproducing the essence of Deubner's Fig. 1. Theoretical dispersion curves
from the Ulrich model are overlaid for comparison.

핵심 결과: 이산 능선을 보여주는 진단적 $k$–$\omega$ 다이어그램으로,
Deubner의 Fig. 1의 본질을 재현한다. Ulrich 모델의 이론적 분산 곡선을 비교를 위해 중첩한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Left panel: k-omega power spectrum (like Deubner's Fig. 1) ---
ax = axes[0]
# Mask zero frequency and zero wavenumber
P_plot = P_smooth.copy()
P_plot[0, :] = 0
P_plot[:, 0] = 0
P_plot[P_plot <= 0] = 1e-10

im = ax.pcolormesh(
    k_pos, freq_pos, P_plot.T,
    cmap="YlOrRd",
    norm=LogNorm(vmin=np.percentile(P_plot[P_plot > 0], 50),
                 vmax=np.percentile(P_plot[P_plot > 0], 99.5)),
    shading="auto",
)
ax.set_xlabel(r"Horizontal wavenumber $k_h$ (Mm$^{-1}$)")
ax.set_ylabel(r"Frequency $\nu$ (mHz)")
ax.set_title("k–ω Power Spectrum (Simulated)\nk–ω 파워 스펙트럼 (시뮬레이션)")
ax.set_xlim(0, 1.2)
ax.set_ylim(0, 8)
plt.colorbar(im, ax=ax, label="Power (log scale)")

# --- Right panel: Same with theoretical curves overlaid ---
ax = axes[1]
im = ax.pcolormesh(
    k_pos, freq_pos, P_plot.T,
    cmap="YlOrRd",
    norm=LogNorm(vmin=np.percentile(P_plot[P_plot > 0], 50),
                 vmax=np.percentile(P_plot[P_plot > 0], 99.5)),
    shading="auto",
)

# Overlay theoretical dispersion curves
colors = ["#0000FF", "#0066FF", "#00AAFF", "#00DDFF", "#00FFAA"]
for n in n_orders:
    kh_th, nu_th = dispersion_curves[n]
    if len(kh_th) > 0:
        ax.plot(kh_th, nu_th, color=colors[n], linewidth=2.5,
                label=f"n = {n}", linestyle="--")

ax.set_xlabel(r"Horizontal wavenumber $k_h$ (Mm$^{-1}$)")
ax.set_ylabel(r"Frequency $\nu$ (mHz)")
ax.set_title("k–ω Diagram + Ulrich Dispersion Curves\nk–ω 다이어그램 + Ulrich 분산 곡선")
ax.set_xlim(0, 1.2)
ax.set_ylim(0, 8)
ax.legend(loc="upper left", fontsize=11, title="Radial order n")
plt.colorbar(im, ax=ax, label="Power (log scale)")

plt.tight_layout()
plt.savefig("15_deubner_1975_k_omega.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: 15_deubner_1975_k_omega.png")

## 7. Phase Velocity Measurement / 위상 속도 측정

Deubner measured the horizontal phase velocity from the slope of the ridges:

Deubner는 능선의 기울기로부터 수평 위상 속도를 측정했다:

$$v_{\text{phase}} = \frac{\omega_{\text{MAX}}}{k_{\text{MAX}}} = \frac{2.1 \times 10^{-2} \text{ s}^{-1}}{0.144 \text{ Mm}^{-1}} = 145 \text{ km s}^{-1}$$

We reproduce this measurement from our simulated k–ω diagram.

시뮬레이션된 k–ω 다이어그램에서 이 측정을 재현한다.

In [ ]:
# Find the peak power location in the k-omega diagram
# Restrict to physically relevant range: k > 0.05, freq in 2-5 mHz (5-min band)
k_mask = k_pos > 0.05
freq_mask = (freq_pos > 2.0) & (freq_pos < 5.0)

P_region = P_smooth[np.ix_(k_mask, freq_mask)]
idx_k, idx_f = np.unravel_index(np.argmax(P_region), P_region.shape)

k_peak = k_pos[k_mask][idx_k]  # Mm^-1
freq_peak = freq_pos[freq_mask][idx_f]  # mHz
omega_peak = freq_peak * 2 * np.pi * 1e-3  # rad/s

v_phase = omega_peak / k_peak  # Mm/s -> km/s
v_phase_km = v_phase * 1e3  # km/s

print("=" * 50)
print("Phase velocity measurement / 위상 속도 측정")
print("=" * 50)
print(f"k_MAX = {k_peak:.3f} Mm^-1")
print(f"ν_MAX = {freq_peak:.2f} mHz (ω_MAX = {omega_peak:.4e} rad/s)")
print(f"v_phase = ω_MAX / k_MAX = {v_phase_km:.0f} km/s")
print(f"\nDeubner's measurement: 145 km/s")
print(f"Our measurement:      {v_phase_km:.0f} km/s")

## 8. Effect of Slit Length on Resolution / 슬릿 길이가 분해능에 미치는 영향

Deubner's key advantage was using a sufficiently long slit. We demonstrate this by
comparing k–ω diagrams from different slit lengths: 50, 150, 300, and 880 arcsec.
The 880 arcsec case corresponds to the improved observation mentioned in the Note.

Deubner의 핵심 이점은 충분히 긴 슬릿을 사용한 것이다. 다른 슬릿 길이(50, 150, 300, 880 arcsec)에서
k–ω 다이어그램을 비교하여 이를 보여준다. 880 arcsec은 Note에 언급된 개선된 관측에 해당한다.

In [ ]:
slit_lengths = [50, 150, 300, 880]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, slit_len in zip(axes, slit_lengths):
    # Use subset of the full simulated data
    nx_sub = min(int(slit_len / SCAN_STEP), nx)

    v_sub = v[:nx_sub, :]
    win_x = np.hamming(nx_sub)
    win_t = np.hamming(nt)
    v_win = v_sub * np.outer(win_x, win_t)

    V_sub = np.fft.fft2(v_win)
    P_sub = np.abs(V_sub) ** 2

    k_sub = np.fft.fftfreq(nx_sub, d=dx_mm)
    k_sub_pos = k_sub[:nx_sub // 2]
    P_sub_pos = P_sub[:nx_sub // 2, :nt // 2]
    P_sub_smooth = gaussian_filter(P_sub_pos, sigma=[1.0, 1.0])
    P_sub_smooth[0, :] = 0
    P_sub_smooth[:, 0] = 0
    P_sub_smooth[P_sub_smooth <= 0] = 1e-10

    im = ax.pcolormesh(
        k_sub_pos, freq_pos, P_sub_smooth.T,
        cmap="YlOrRd",
        norm=LogNorm(
            vmin=np.percentile(P_sub_smooth[P_sub_smooth > 0], 50),
            vmax=np.percentile(P_sub_smooth[P_sub_smooth > 0], 99.5),
        ),
        shading="auto",
    )

    # Overlay theory
    for n in n_orders:
        kh_th, nu_th = dispersion_curves[n]
        if len(kh_th) > 0:
            ax.plot(kh_th, nu_th, "b--", linewidth=1, alpha=0.6)

    ax.set_xlim(0, 1.2)
    ax.set_ylim(0, 8)
    ax.set_title(f"Slit = {slit_len}\"")
    ax.set_xlabel(r"$k_h$ (Mm$^{-1}$)")
    if slit_len == 50:
        ax.set_ylabel(r"$\nu$ (mHz)")

fig.suptitle(
    "Effect of Slit Length on k–ω Resolution\n슬릿 길이에 따른 k–ω 분해능 변화",
    fontsize=15, y=1.02,
)
plt.tight_layout()
plt.show()

## 9. Summary / 요약

### What we demonstrated / 시연한 내용

1. **Solar model**: Built a polytropic sound speed profile for the convection zone
   태양 모델: 대류층에 대한 polytrope 음속 프로파일 구축

2. **Dispersion curves**: Numerically solved the resonance condition to obtain theoretical p-mode ridges $\omega(\ell, n)$ for $n = 0, 1, 2, 3, 4$
   분산 곡선: 공진 조건을 수치적으로 풀어 이론적 p-mode 능선 $\omega(\ell, n)$ 도출

3. **Simulated observation**: Generated a synthetic velocity field $v(x, t)$ matching Deubner's observational setup (300" slit, 110 s cadence, ~4 hours)
   관측 시뮬레이션: Deubner의 관측 설정에 맞는 합성 속도장 $v(x, t)$ 생성

4. **2D FFT → k–ω diagram**: Reproduced the diagnostic diagram showing discrete ridges — the key result of Deubner (1975)
   2D FFT → k–ω 다이어그램: 이산 능선을 보여주는 진단 다이어그램 재현 — Deubner (1975)의 핵심 결과

5. **Phase velocity**: Measured $v_{\text{phase}}$ from the power spectrum peak, consistent with Deubner's 145 km/s
   위상 속도: 파워 스펙트럼 피크에서 $v_{\text{phase}}$ 측정, Deubner의 145 km/s와 일관

6. **Resolution effect**: Demonstrated that longer slits produce sharper ridge separation — explaining why Deubner succeeded with 300" while previous observers failed with shorter coverage
   분해능 효과: 더 긴 슬릿이 더 선명한 능선 분리를 생성함을 시연 — Deubner가 300"로 성공한 반면 이전 관측자들이 짧은 커버리지로 실패한 이유 설명